In [1]:

import psycopg2

try:
    # Connect to PostgreSQL
    conn = psycopg2.connect(
        host="localhost",      # or "127.0.0.1"
        port="5432",           # default PostgreSQL port
        database="postgres",
        user="postgres",
        password="attackontitan7"
    )

    # Create a cursor to execute SQL commands
    cur = conn.cursor()

    # Example query
    cur.execute("SELECT version();")

    # Fetch result
    db_version = cur.fetchone()
    print("Connected to:", db_version)

    # Clean up
    cur.close()
    conn.close()
except Exception as e:
    print("Error")


Connected to: ('PostgreSQL 17.5 on x86_64-windows, compiled by msvc-19.44.35209, 64-bit',)


In [ ]:
import psycopg2

class TravelDBPrimaryBackup:
    def __init__(self):
        self.primary_conn = psycopg2.connect(
            dbname="travel_primary",
            user="postgres",
            password="your_password",
            host="localhost",
            port="5432"
        )
        self.backup_conn = psycopg2.connect(
            dbname="travel_backup",
            user="postgres",
            password="your_password",
            host="localhost",
            port="5433"   # backup can run on another port
        )
        self.primary_crashed = False
        self._setup_db(self.primary_conn)
        self._setup_db(self.backup_conn)
        print("[+] Primary and Backup PostgreSQL databases initialized.")

    def _setup_db(self, conn):
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS restaurants (
                id SERIAL PRIMARY KEY,
                name TEXT,
                location TEXT,
                price INTEGER
            )
        """)
        conn.commit()

    def insert_initial_data(self):
        data = [
            ("XYZ Cafe", "Mumbai", 200),
            ("Royal Palace", "Pune", 600),
            ("Techie Tiffins", "Bangalore", 300),
            ("Spice Hub", "Delhi", 350)
        ]
        cursor = self.primary_conn.cursor()
        cursor.executemany("INSERT INTO restaurants (name, location, price) VALUES (%s, %s, %s)", data)
        self.primary_conn.commit()
        self.replicate_to_backup()
        print("[✓] Data inserted and replicated.")

    def replicate_to_backup(self):
        # manual replication
        cursor_p = self.primary_conn.cursor()
        cursor_b = self.backup_conn.cursor()
        cursor_p.execute("SELECT name, location, price FROM restaurants")
        rows = cursor_p.fetchall()
        cursor_b.execute("DELETE FROM restaurants")
        cursor_b.executemany("INSERT INTO restaurants (name, location, price) VALUES (%s, %s, %s)", rows)
        self.backup_conn.commit()
        print("[↺] Backup synchronized with primary.")

    def query_restaurants(self, location):
        conn = self.backup_conn if self.primary_crashed else self.primary_conn
        cursor = conn.cursor()
        cursor.execute("SELECT name, location, price FROM restaurants WHERE location = %s", (location,))
        return cursor.fetchall()